# Step 01: BPE tokenizer

Tests for `01-tokenization/bpe.py`. Write code in the `.py` file, only import and test here.

Run the cells top to bottom (Run All).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("01-tokenization")

from bpe import get_pair_counts, merge, BPETokenizer

## 1. `get_pair_counts`

Count how often each adjacent pair occurs.

In [2]:
get_pair_counts([1, 2, 3, 1, 2])
# expect {(1, 2): 2, (2, 3): 1, (3, 1): 1}

{(1, 2): 2, (2, 3): 1, (3, 1): 1}

## 2. `merge`

Replace a pair with a new token id, left to right.

In [3]:
print(merge([1, 2, 3, 1, 2], (1, 2), 99))  # expect [99, 3, 99]
print(merge([1, 1, 1], (1, 1), 99))        # expect [99, 1] (overlap: only the first pair merges)

[99, 3, 99]
[99, 1]


## 3. `train`

Learn merges: count pairs, merge the most frequent, repeat.

In [4]:
tok = BPETokenizer()
tok.train("aaabdaaabac", vocab_size=258)
tok.merges  # expect {(97, 97): 256, (256, 97): 257}

{(97, 97): 256, (256, 97): 257}

## 4. `encode` and `decode`

Encode replays the learned merges in order. Decode maps ids back to bytes.

In [5]:
ids = tok.encode("aaab")
print(ids)              # expect [257, 98]
print(tok.decode(ids))  # expect 'aaab'

[257, 98]
aaab


## 5. Real text and tiktoken

Train on a real file, check the round trip, and compare with GPT-2's tokenizer. Ours uses only 44 merges from one small file, so it needs more tokens than GPT-2's ~50k vocab.

In [6]:
# train on a real file (bpe.py itself), then see which chunks it learned
text = open("01-tokenization/bpe.py").read()
real = BPETokenizer()
real.train(text, vocab_size=300)

sample = "def merge(ids): return ids"
ids = real.encode(sample)
print(len(sample.encode()), "bytes ->", len(ids), "tokens")
print("round trip:", real.decode(ids) == sample)  # expect True

# show the learned tokens as readable text
vocab = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in real.merges.items():
    vocab[new_id] = vocab[a] + vocab[b]
print([vocab[t] for t in ids])

26 bytes -> 15 tokens
round trip: True
[b'de', b'f', b' ', b'merge', b'(ids', b')', b': ', b'r', b'e', b't', b'u', b'r', b'n', b' ', b'ids']


In [7]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")  # real GPT-2 tokenizer, ~50k vocab

test = "hello world, this is a tokenizer test"
print("mine:    ", len(real.encode(test)), "tokens")
print("tiktoken:", len(enc.encode(test)), "tokens")
print([enc.decode([t]) for t in enc.encode(test)])  # GPT-2's tokens as readable chunks

mine:     27 tokens
tiktoken: 9 tokens
['hello', ' world', ',', ' this', ' is', ' a', ' token', 'izer', ' test']


In [8]:
# edge cases: empty, single char, accents, emoji (works because we start from bytes)
for s in ["", "a", "héllo 🙂"]:
    print(repr(s), real.decode(real.encode(s)) == s)  # expect True for all

'' True
'a' True
'héllo 🙂' True
